In [1]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your new DBpedia test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "sparqlce_dbpedia", "best_model.pt")

TOPKS = [1, 3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")
# Enable per-bin lexical stratification for every model below:
#   [0,0.3) adversarial | [0.3,0.5) divergent | [0.5,0.8) moderate | [0.8,1.0) near-identical
import evaluate_rq1
evaluate_rq1.STRATIFY_BINS = [0.3, 0.5, 0.8]  # cumulative: cos < threshold


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Success! Project root set to: /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings
Dataset Loaded -> Test Split: 462
Evaluation Ready -> Anchors: 462 | Corpus: 1841


In [2]:
# E5-base-v2 (fine-tuned on SQuARE-DBpedia) — capacity-matched trained baseline
# Checkpoint saved by 04_e5_contrastive_finetune_square.ipynb
E5_FT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_dbpedia", "best_model")
evaluate_sentence_transformer(
    E5_FT_PATH,
    "E5-base-v2 (fine-tuned)",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    transform=lambda q: f"query: {q}",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8196.10it/s]



E5-base-v2 (fine-tuned)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.5801  0.3419  0.3706  0.5801  0.5801    0.5801  0.5801
@3     0.3773  0.4503  0.3113  0.6497  0.7359    0.3773  0.5928
@5     0.3117  0.4997  0.2795  0.6637  0.7965    0.3117  0.6007
@10    0.2442  0.5794  0.2520  0.6716  0.8615    0.2442  0.6103
@20    0.1896  0.6811  0.2336  0.6749  0.9069    0.1896  0.6236
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.1491   0.1915   0.2016   0.2118   0.2191
            <0.5    0.2787      0.2921      187   0.3957   0.4545   0.4679   0.4763   0.4808
            <0.8    0.4967      0.5867      376   0.4761   0.5638   0.5816   0.5914   0.5954


In [3]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# BM25
evaluate_bm25(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen2.5-Coder (generative code LLM used as an embedder, last-token pooling;
# 0.5B ≈ scale-matched to the dense 0.6B Qwen3-Embedding it replaces)
evaluate_code_llm('Qwen/Qwen2.5-Coder-0.5B', 'Qwen2.5-Coder-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

evaluate_code_llm('Qwen/Qwen3-0.6B', 'Qwen3-0.6B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/checkpoints/sparqlce_dbpedia/best_model.pt...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6485.47it/s]



SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.6861  0.3284  0.3700  0.6861  0.6861    0.6861  0.6861
@3     0.4747  0.4525  0.3435  0.7323  0.7879    0.4747  0.6754
@5     0.4017  0.5089  0.3245  0.7369  0.8074    0.4017  0.6763
@10    0.3232  0.6091  0.3144  0.7426  0.8506    0.3232  0.6801
@20    0.2522  0.7419  0.3023  0.7450  0.8874    0.2522  0.6926
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.2456   0.3363   0.3463   0.3598   0.3690
            <0.5    0.2787      0.2921      187   0.5348   0.6141   0.6202   0.6274   0.6321
            <0.8    0.4967      0.5867      376   0.6941   0.7504   0.7555   0.7622   0.7640

Baseline: TF-IDF
Binary relevance (annotated 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7193.73it/s]



E5-base-v2
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.3355  0.2093  0.2276  0.3355  0.3355    0.3355  0.3355
@3     0.1970  0.2782  0.1842  0.3831  0.4394    0.1970  0.3354
@5     0.1468  0.2976  0.1513  0.3911  0.4740    0.1468  0.3332
@10    0.1035  0.3315  0.1205  0.3997  0.5390    0.1035  0.3358
@20    0.0751  0.3763  0.0998  0.4063  0.6342    0.0751  0.3426
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0263   0.0468   0.0468   0.0504   0.0566
            <0.5    0.2787      0.2921      187   0.0963   0.1399   0.1490   0.1552   0.1612
            <0.8    0.4967      0.5867      376   0.1835   0.2407   0.2511   0.2615   0.2697


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 8692.92it/s]



Qwen2.5-Coder-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.2100  0.0939  0.1083  0.2100  0.2100    0.2100  0.2100
@3     0.1450  0.1513  0.1108  0.2594  0.3203    0.1450  0.2085
@5     0.1173  0.1771  0.1030  0.2701  0.3658    0.1173  0.2094
@10    0.0855  0.2231  0.0914  0.2807  0.4437    0.0855  0.2136
@20    0.0630  0.2806  0.0809  0.2878  0.5519    0.0630  0.2206
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0351   0.0439   0.0439   0.0453   0.0470
            <0.5    0.2787      0.2921      187   0.0963   0.1203   0.1251   0.1317   0.1365
            <0.8    0.4967      0.5867      376   0.1676   0.2176   0.2280   0.2372   0.2452


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 8670.31it/s]
[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Qwen3-0.6B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.2251  0.0883  0.1037  0.2251  0.2251    0.2251  0.2251
@3     0.1537  0.1580  0.1146  0.2742  0.3442    0.1537  0.2171
@5     0.1286  0.1989  0.1121  0.2886  0.4069    0.1286  0.2224
@10    0.0955  0.2473  0.1011  0.2975  0.4740    0.0955  0.2254
@20    0.0693  0.2822  0.0872  0.3016  0.5368    0.0693  0.2257
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0351   0.0424   0.0446   0.0473   0.0499
            <0.5    0.2787      0.2921      187   0.1176   0.1462   0.1523   0.1573   0.1611
            <0.8    0.4967      0.5867      376   0.1995   0.2438   0.2551   0.2647   0.2697


In [4]:
# Prior-literature SPARQL / code encoder baselines (zero-shot, mean-pooled):
#   CodeBERT (SparqlCE's own backbone), GraphCodeBERT (CodeBERT + data-flow
#   graph), SPBERT (BERT pretrained on SPARQL). Defined in evaluate_rq1.py as
#   SPARQL_ENCODER_BASELINES. The "mean pooling" notice each emits is correct
#   here (raw MLM encoders), not the LoRA-adapter bug.
evaluate_sparql_encoder_baselines(eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37168.97it/s]



CodeBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.2532  0.1326  0.1469  0.2532  0.2532    0.2532  0.2532
@3     0.1623  0.1887  0.1302  0.2980  0.3528    0.1623  0.2511
@5     0.1286  0.2131  0.1173  0.3109  0.4069    0.1286  0.2498
@10    0.0913  0.2416  0.0981  0.3198  0.4719    0.0913  0.2469
@20    0.0680  0.2870  0.0867  0.3248  0.5455    0.0680  0.2520
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0263   0.0439   0.0439   0.0439   0.0469
            <0.5    0.2787      0.2921      187   0.1230   0.1497   0.1575   0.1629   0.1663
            <0.8    0.4967      0.5867      376   0.1676   0.2123   0.2281   0.2388   0.2447


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 41887.76it/s]
[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



GraphCodeBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.2619  0.1394  0.1539  0.2619  0.2619    0.2619  0.2619
@3     0.1681  0.2019  0.1383  0.3081  0.3680    0.1681  0.2618
@5     0.1320  0.2209  0.1207  0.3167  0.4048    0.1320  0.2590
@10    0.0907  0.2496  0.0985  0.3240  0.4589    0.0907  0.2547
@20    0.0649  0.2987  0.0838  0.3290  0.5303    0.0649  0.2590
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0351   0.0395   0.0395   0.0443   0.0466
            <0.5    0.2787      0.2921      187   0.1390   0.1711   0.1746   0.1800   0.1839
            <0.8    0.4967      0.5867      376   0.1888   0.2332   0.2437   0.2526   0.2579


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 40969.75it/s]
[transformers] BertModel LOAD REPORT from: razent/spbert-mlm-wso-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consid


SPBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.2511  0.1266  0.1435  0.2511  0.2511    0.2511  0.2511
@3     0.1717  0.1949  0.1391  0.3077  0.3810    0.1717  0.2548
@5     0.1312  0.2204  0.1224  0.3170  0.4221    0.1312  0.2516
@10    0.0978  0.2645  0.1067  0.3282  0.5087    0.0978  0.2570
@20    0.0736  0.3163  0.0941  0.3335  0.5844    0.0736  0.2644
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0351   0.0468   0.0503   0.0513   0.0513
            <0.5    0.2787      0.2921      187   0.1123   0.1488   0.1523   0.1588   0.1630
            <0.8    0.4967      0.5867      376   0.1862   0.2460   0.2556   0.2677   0.2741


In [5]:

display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5931.89it/s]



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.2987  0.1669  0.1836  0.2987  0.2987    0.2987  0.2987
@3     0.1854  0.2401  0.1621  0.3492  0.4177    0.1854  0.2973
@5     0.1433  0.2745  0.1426  0.3580  0.4567    0.1433  0.2991
@10    0.0994  0.3132  0.1142  0.3658  0.5130    0.0994  0.3006
@20    0.0746  0.3705  0.0984  0.3726  0.6061    0.0746  0.3107
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0526   0.0629   0.0646   0.0670   0.0698
            <0.5    0.2787      0.2921      187   0.1337   0.1533   0.1602   0.1653   0.1697
            <0.8    0.4967      0.5867      376   0.1941   0.2451   0.2538   0.2623   0.2697


In [6]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7986.63it/s]



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@1     0.3290  0.1940  0.2113  0.3290  0.3290    0.3290  0.3290
@3     0.1898  0.2555  0.1704  0.3723  0.4329    0.1898  0.3180
@5     0.1606  0.2884  0.1559  0.3906  0.5130    0.1606  0.3271
@10    0.1134  0.3295  0.1272  0.3982  0.5693    0.1134  0.3294
@20    0.0785  0.3763  0.1031  0.4021  0.6277    0.0785  0.3332
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@1    MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0439   0.0497   0.0550   0.0574   0.0615
            <0.5    0.2787      0.2921      187   0.1176   0.1453   0.1592   0.1647   0.1675
            <0.8    0.4967      0.5867      376   0.1995   0.2478   0.2704   0.2789   0.2835
